# Q5 Kaggle — Clean Pipeline

Single-pass notebook from raw data to all submissions. No re-tuning, no failed experiments — just the path that works.

**Outputs:**
- `submissions/lgbm_v4_tuned.csv` (LightGBM, Optuna-tuned)
- `submissions/ensemble_lgb_cb.csv` (LGB + CatBoost blend)
- `submissions/blend_with_gpt.csv` (above + GPT submission)
- `submissions/pseudolabel_blend.csv` (pseudo-labeling stacked)

**Runtime:** ~10 minutes total.


## 1. Setup

In [1]:
from pathlib import Path
import warnings; warnings.filterwarnings("ignore", category=UserWarning)
import re

import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 2025
HERE = Path.cwd()
REPO = HERE.parent if HERE.name == "q5_kaggle" else HERE
DATA_DIR = REPO / "Assignment1" / "cs-610-assignment-1-question-5-2026"
SUB_DIR = (HERE if HERE.name == "q5_kaggle" else HERE / "q5_kaggle") / "submissions"
SUB_DIR.mkdir(exist_ok=True, parents=True)
UPLOAD_DIR = REPO.parent / "uploads"  # GPT submission lives here

# Best LightGBM params from earlier 50-trial Optuna run on v3 features
BEST_PARAMS = {
    "learning_rate":    0.023244641257268547,
    "num_leaves":       90,
    "min_data_in_leaf": 14,
    "feature_fraction": 0.5042299812794246,
    "bagging_fraction": 0.9708522995554573,
    "bagging_freq":     4,
    "lambda_l1":        1.4345478190845018e-07,
    "lambda_l2":        3.437058206065732,
}

print("Setup OK")


Setup OK


In [ ]:
# Override: GPT submission lives in submissions/ on this machine
GPT_PATH = SUB_DIR / "sub_lgb_base_plus.csv"
print(f"GPT path: {GPT_PATH}")
print(f"exists: {GPT_PATH.exists()}")

## 2. Load data

In [2]:
train = pd.read_csv(DATA_DIR / "train.csv")
test  = pd.read_csv(DATA_DIR / "test.csv")
y_train = train["target"].values
print(f"train: {train.shape}  test: {test.shape}  bot rate: {y_train.mean():.3f}")


train: (26206, 19)  test: (11232, 19)  bot rate: 0.335


## 3. Feature engineering

All the engineering that survived our experiments — basics, ratios, temporal, bio structural counts.


In [3]:
TOP_LANGS = train["lang"].value_counts().head(8).index.tolist()

EMOJI_RE = re.compile(
    "[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F\U0001F1E0-\U0001F1FF\U00002700-\U000027BF"
    "\U00002600-\U000026FF]"
)

def featurize(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)

    # Counts (raw + log1p)
    counts = ["favourites_count", "followers_count", "friends_count",
              "statuses_count", "average_tweets_per_day", "account_age_days"]
    for c in counts: out[c] = df[c].astype(float)
    for c in ["favourites_count", "followers_count", "friends_count", "statuses_count"]:
        out[f"log_{c}"] = np.log1p(out[c])

    # Booleans
    for c in ["default_profile", "default_profile_image", "geo_enabled", "verified"]:
        out[c] = df[c].astype(int)

    # Presence flags + length features
    out["has_description"] = df["description"].notna().astype(int)
    out["has_location"]    = (df["location"].notna() & (df["location"] != "unknown")).astype(int)
    out["has_url_bg"]      = df["profile_background_image_url"].notna().astype(int)
    out["desc_len"]  = df["description"].fillna("").str.len()
    out["sn_len"]    = df["screen_name"].fillna("").str.len()
    out["sn_digits"] = (df["screen_name"].fillna("").str.count(r"\d") / out["sn_len"].clip(lower=1))

    # Categorical
    lang = df["lang"].fillna("MISSING")
    out["lang"] = lang.where(lang.isin(TOP_LANGS), "OTHER")

    # Ratios
    out["followers_per_friend"] = df["followers_count"] / df["friends_count"].clip(lower=1)
    out["statuses_per_day"]     = df["statuses_count"]  / df["account_age_days"].clip(lower=1)

    # Temporal
    ct = pd.to_datetime(df["created_at"])
    out["created_year"]  = ct.dt.year.astype(float)
    out["created_month"] = ct.dt.month.astype(float)
    out["created_dow"]   = ct.dt.dayofweek.astype(float)
    out["created_hour"]  = ct.dt.hour.astype(float)
    out["hour_sin"] = np.sin(2*np.pi*out["created_hour"]/24)
    out["hour_cos"] = np.cos(2*np.pi*out["created_hour"]/24)
    out["dow_sin"]  = np.sin(2*np.pi*out["created_dow"]/7)
    out["dow_cos"]  = np.cos(2*np.pi*out["created_dow"]/7)

    # Bio structural
    s = df["description"].fillna("")
    n = s.str.len().clip(lower=1)
    out["bio_url_count"]      = s.str.count(r"https?://")
    out["bio_hashtag_count"]  = s.str.count(r"#\w+")
    out["bio_mention_count"]  = s.str.count(r"@\w+")
    out["bio_emoji_count"]    = s.apply(lambda x: len(EMOJI_RE.findall(x)))
    out["bio_caps_ratio"]     = s.str.count(r"[A-Z]") / n
    out["bio_digit_ratio"]    = s.str.count(r"\d") / n
    out["bio_special_ratio"]  = s.str.count(r"[^\w\s]") / n
    out["bio_word_count"]     = s.str.split().apply(len)

    return out

X_train = featurize(train)
X_test  = featurize(test)
print(f"feature matrix: train {X_train.shape}, test {X_test.shape}")


feature matrix: train (26206, 39), test (11232, 39)


## 4. Build sparse matrix (numerics + lang one-hot + TF-IDF)


In [4]:
numeric_cols = [c for c in X_train.columns if c != "lang"]
print(f"{len(numeric_cols)} dense numeric/binary cols + 1 categorical (lang)")

# TF-IDF on description (word-level, fits on train only to avoid leakage)
tfidf = TfidfVectorizer(
    max_features=500, ngram_range=(1, 2),
    min_df=5, max_df=0.95, lowercase=True,
    sublinear_tf=True, strip_accents="unicode",
)
Xt_train = tfidf.fit_transform(train["description"].fillna(""))
Xt_test  = tfidf.transform(test["description"].fillna(""))

# One-hot encode lang
oh = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
lang_train_oh = oh.fit_transform(X_train[["lang"]])
lang_test_oh  = oh.transform(X_test[["lang"]])

# Numeric matrix as sparse, then hstack everything
num_train = sparse.csr_matrix(X_train[numeric_cols].fillna(0).values)
num_test  = sparse.csr_matrix(X_test[numeric_cols].fillna(0).values)

X_sp_train = sparse.hstack([num_train, lang_train_oh, Xt_train]).tocsr()
X_sp_test  = sparse.hstack([num_test,  lang_test_oh,  Xt_test ]).tocsr()
print(f"sparse: train {X_sp_train.shape}, test {X_sp_test.shape}")


38 dense numeric/binary cols + 1 categorical (lang)
sparse: train (26206, 547), test (11232, 547)


## 5. CV helper that returns OOF + fold-averaged test predictions


In [5]:
def lgbm_cv(X, y, X_test, n_splits=5, params=None, verbose=True):
    """5-fold CV with LightGBM. Returns OOF predictions and fold-averaged test predictions."""
    p = dict(n_estimators=2000, n_jobs=-1, verbose=-1, random_state=RANDOM_STATE)
    if params: p.update(params)

    take = lambda M, idx: M.iloc[idx] if hasattr(M, "iloc") else M[idx]
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    aucs, iters = [], []
    oof = np.zeros(X.shape[0])
    test_preds = np.zeros((n_splits, X_test.shape[0]))

    for fold, (tr, va) in enumerate(skf.split(np.zeros(X.shape[0]), y)):
        clf = lgb.LGBMClassifier(**p)
        clf.fit(take(X, tr), y[tr],
                eval_set=[(take(X, va), y[va])], eval_metric="auc",
                callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
        oof[va] = clf.predict_proba(take(X, va))[:, 1]
        test_preds[fold] = clf.predict_proba(X_test)[:, 1]
        aucs.append(roc_auc_score(y[va], oof[va]))
        iters.append(clf.best_iteration_)
        if verbose:
            print(f"  Fold {fold+1}: AUC={aucs[-1]:.4f}  iter={iters[-1]}")

    if verbose:
        print(f"  Mean fold AUC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
        print(f"  OOF AUC      : {roc_auc_score(y, oof):.4f}")
    return {"oof": oof, "test_avg": test_preds.mean(axis=0),
            "aucs": aucs, "iters": iters}


## 6. Train LightGBM v4 (Optuna-tuned)

OOF should be ≈ 0.945+. Best public LB so far: 0.93998.


In [6]:
print("=== LightGBM v4 (5-fold CV) ===")
lgb_result = lgbm_cv(X_sp_train, y_train, X_sp_test, params=BEST_PARAMS)
oof_lgb = lgb_result["oof"]
test_lgb = lgb_result["test_avg"]

# Also write a fold-averaged submission (this is our "v4_tuned")
sub = pd.DataFrame({"index": test["index"].values, "target": test_lgb})
sub.to_csv(SUB_DIR / "lgbm_v4_tuned.csv", index=False, float_format="%.6f")
print(f"\nWrote: {SUB_DIR/'lgbm_v4_tuned.csv'}")


=== LightGBM v4 (5-fold CV) ===
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[342]	valid_0's auc: 0.947992	valid_0's binary_logloss: 0.267066
  Fold 1: AUC=0.9480  iter=342
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[402]	valid_0's auc: 0.942531	valid_0's binary_logloss: 0.277085
  Fold 2: AUC=0.9425  iter=402
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[287]	valid_0's auc: 0.938785	valid_0's binary_logloss: 0.289054
  Fold 3: AUC=0.9388  iter=287
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[325]	valid_0's auc: 0.948051	valid_0's binary_logloss: 0.265003
  Fold 4: AUC=0.9481  iter=325
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[463]	valid_0's auc: 0.948061	valid_0's binary_logloss: 0.266524
  Fold 5: AUC=0.9481  iter=463
  Mean f

## 7. Train CatBoost on dense features (different algorithm = ensemble diversity)

CatBoost OOF lands ≈ 0.940. Lower individual AUC but correlation 0.978 with LightGBM — enough room to lift a blend.


In [7]:
def cb_cv(X, y, X_test, n_splits=5, verbose=True):
    """5-fold CV for CatBoost on dense DataFrame with native categorical (lang)."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    aucs, iters = [], []
    oof = np.zeros(len(X))
    test_preds = np.zeros((n_splits, len(X_test)))

    for fold, (tr, va) in enumerate(skf.split(np.zeros(len(X)), y)):
        clf = CatBoostClassifier(
            iterations=2000, learning_rate=0.05, depth=6, l2_leaf_reg=3.0,
            loss_function="Logloss", eval_metric="AUC",
            random_seed=RANDOM_STATE, early_stopping_rounds=50, verbose=False,
        )
        clf.fit(X.iloc[tr], y[tr], eval_set=(X.iloc[va], y[va]),
                cat_features=["lang"], use_best_model=True)
        oof[va] = clf.predict_proba(X.iloc[va])[:, 1]
        test_preds[fold] = clf.predict_proba(X_test)[:, 1]
        aucs.append(roc_auc_score(y[va], oof[va]))
        iters.append(clf.best_iteration_)
        if verbose:
            print(f"  Fold {fold+1}: AUC={aucs[-1]:.4f}  iter={iters[-1]}")

    if verbose:
        print(f"  Mean fold AUC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
        print(f"  OOF AUC      : {roc_auc_score(y, oof):.4f}")
    return {"oof": oof, "test_avg": test_preds.mean(axis=0)}

print("=== CatBoost (5-fold CV) ===")
cb_result = cb_cv(X_train, y_train, X_test)
oof_cb = cb_result["oof"]
test_cb = cb_result["test_avg"]

print(f"\ncorr(LGB OOF, CB OOF): {np.corrcoef(oof_lgb, oof_cb)[0,1]:.4f}")


=== CatBoost (5-fold CV) ===
  Fold 1: AUC=0.9429  iter=693
  Fold 2: AUC=0.9400  iter=980
  Fold 3: AUC=0.9357  iter=488
  Fold 4: AUC=0.9428  iter=632
  Fold 5: AUC=0.9438  iter=554
  Mean fold AUC: 0.9410 ± 0.0029
  OOF AUC      : 0.9410

corr(LGB OOF, CB OOF): 0.9827


## 8. Find best LGB/CB blend weight, save ensemble submission


In [8]:
weights = np.arange(0.5, 1.01, 0.02)
results = [(w, roc_auc_score(y_train, w*oof_lgb + (1-w)*oof_cb)) for w in weights]
best_w, best_auc = max(results, key=lambda x: x[1])
print(f"Best LGB/CB blend on OOF: w_lgb={best_w:.2f}  →  OOF {best_auc:.4f}")
print(f"vs LightGBM alone:        OOF {roc_auc_score(y_train, oof_lgb):.4f}")

ensemble_test = best_w * test_lgb + (1 - best_w) * test_cb
sub = pd.DataFrame({"index": test["index"].values, "target": ensemble_test})
sub.to_csv(SUB_DIR / "ensemble_lgb_cb.csv", index=False, float_format="%.6f")
print(f"\nWrote: {SUB_DIR/'ensemble_lgb_cb.csv'}")


Best LGB/CB blend on OOF: w_lgb=0.80  →  OOF 0.9453
vs LightGBM alone:        OOF 0.9450

Wrote: C:\Users\user\Desktop\6. AML\AML_rep\q5_kaggle\submissions\ensemble_lgb_cb.csv


## 9. Blend with GPT's submission

Loads GPT's `sub_lgb_base_plus.csv` (public 0.94211) and blends with our ensemble for diversity. Weights chosen with bias toward GPT since its public was higher.


In [12]:
GPT_PATH = SUB_DIR / "sub_lgb_base_plus.csv"
if GPT_PATH.exists():
    gpt = pd.read_csv(GPT_PATH).set_index("index")["target"]
    # Reindex to match our test order
    gpt_aligned = gpt.reindex(test["index"].values).values

    print(f"corr(ensemble, gpt): {np.corrcoef(ensemble_test, gpt_aligned)[0,1]:.4f}")
    print(f"corr(lgb_v4 , gpt): {np.corrcoef(test_lgb,    gpt_aligned)[0,1]:.4f}")

    # Weighted blend — bias toward GPT (its public was 0.94211 vs our 0.93998)
    blend = 0.55 * gpt_aligned + 0.30 * ensemble_test + 0.15 * test_lgb
    sub = pd.DataFrame({"index": test["index"].values, "target": blend})
    sub.to_csv(SUB_DIR / "blend_with_gpt.csv", index=False, float_format="%.6f")
    print(f"\nWrote: {SUB_DIR/'blend_with_gpt.csv'}")
else:
    print(f"GPT submission not found at {GPT_PATH} — skipping this section")
    blend = None


corr(ensemble, gpt): 0.9781
corr(lgb_v4 , gpt): 0.9762

Wrote: C:\Users\user\Desktop\6. AML\AML_rep\q5_kaggle\submissions\blend_with_gpt.csv


## 10. Pseudo-labeling

Take confident test predictions from the GPT-blend, append them to training data, refit LightGBM. Standard +0.001-0.003 trick.


In [13]:
if blend is not None:
    pl_source = blend
else:
    pl_source = ensemble_test

# Confident predictions only (top 90% / bottom 10%)
HIGH, LOW = 0.90, 0.10
mask_high = pl_source > HIGH
mask_low  = pl_source < LOW
mask_pl = mask_high | mask_low
print(f"Pseudo-labeling {mask_pl.sum()} of {len(pl_source)} test rows ({mask_pl.mean()*100:.1f}%)")
print(f"  high-confidence bots: {mask_high.sum()}")
print(f"  high-confidence humans: {mask_low.sum()}")

# Build augmented training set
X_pl = X_sp_test[mask_pl]
y_pl = (pl_source[mask_pl] > 0.5).astype(int)

X_aug = sparse.vstack([X_sp_train, X_pl]).tocsr()
y_aug = np.concatenate([y_train, y_pl])
print(f"Augmented training: {X_aug.shape}")

# Refit LightGBM with augmented data + best params, fixed n_estimators ~ what we saw in CV
clf_pl = lgb.LGBMClassifier(
    n_estimators=int(np.mean(lgb_result["iters"]) * 1.1),  # slightly more trees for bigger dataset
    n_jobs=-1, verbose=-1, random_state=RANDOM_STATE, **BEST_PARAMS,
)
clf_pl.fit(X_aug, y_aug)
test_pl = clf_pl.predict_proba(X_sp_test)[:, 1]
print(f"\nPseudo-labeled model test pred mean: {test_pl.mean():.4f}")


Pseudo-labeling 7045 of 11232 test rows (62.7%)
  high-confidence bots: 1812
  high-confidence humans: 5233
Augmented training: (33251, 547)

Pseudo-labeled model test pred mean: 0.3183


## 11. Final submission: pseudo-label + ensemble + GPT blend


In [14]:
if blend is not None:
    final = 0.40 * test_pl + 0.35 * gpt_aligned + 0.15 * test_lgb + 0.10 * test_cb
else:
    final = 0.5 * test_pl + 0.35 * test_lgb + 0.15 * test_cb

sub = pd.DataFrame({"index": test["index"].values, "target": final})
sub.to_csv(SUB_DIR / "pseudolabel_blend.csv", index=False, float_format="%.6f")
print(f"Wrote: {SUB_DIR/'pseudolabel_blend.csv'}")
print(f"Mean predicted bot rate: {final.mean():.4f}")
sub.head()


Wrote: C:\Users\user\Desktop\6. AML\AML_rep\q5_kaggle\submissions\pseudolabel_blend.csv
Mean predicted bot rate: 0.3222


,index,target
0,0,0.007798
1,1,0.861853
2,2,0.077660
3,3,0.038529
4,4,0.994002


## 12. Summary

| File | What it is |
|---|---|
| `lgbm_v4_tuned.csv` | LightGBM with Optuna-tuned params, fold-averaged |
| `ensemble_lgb_cb.csv` | LGB + CatBoost weighted blend |
| `blend_with_gpt.csv` | Above + GPT submission, GPT-weighted |
| `pseudolabel_blend.csv` | Pseudo-labeled LGB + GPT + ensemble blend |

**Submission strategy:** save 1 slot per day for `pseudolabel_blend.csv`. Pick final 2 by public LB at deadline, prioritizing diversity (e.g., `blend_with_gpt.csv` + `pseudolabel_blend.csv`).
